# About Dataset

## Context

The Lord of the Rings is a film series consisting of three high fantasy adventure films directed by Peter Jackson. They are based on the novel The Lord of the Rings by J. R. R. Tolkien. The films are subtitled The Fellowship of the Ring (2001), The Two Towers (2002) and The Return of the King (2003).
Source: https://en.wikipedia.org/wiki/The_Lord_of_the_Rings_(film_series)

## Content
lotr_scripts.csv -- Scripts from the LOTR movies

lotr_characters.csv -- Descriptions of Characters from the LOTR movies

In [57]:
# Import Libraries
import pandas as pd
import nltk

In [58]:
df = pd.read_csv("lotr_characters.csv")
rt = pd.read_csv("lotr_scripts.csv")

In [59]:
df.sample(5)

,birth,death,gender,hair,height,name,race,realm,spouse
467,NaN,NaN,NaN,NaN,NaN,Gorgol,NaN,NaN,NaN
523,SA 575,NaN,Male,NaN,NaN,Cemendur (son of Axantur),Men,NaN,Unnamed wife
632,TA 895,TA 1110,Male,NaN,NaN,Mallor,Men,"Arthedain,Arnor",Unnamed wife
21,TA 654,TA 913,Male,NaN,NaN,Tarannon Falastur,Men,Gondor,Berúthiel
207,FO 43,"Before ,FO 220",Male,Dark,NaN,Eldarion,Men,"Reunited Kingdom,Arnor,Gondor",NaN


In [60]:
rt.sample(5)

,Unnamed: 0,char,dialog,movie
958,958,SAM,"Because.Because that's what he is, Mr. Frodo.T...",The Two Towers
1348,1348,PIPPIN,Merry!,The Two Towers
211,211,DENETHOR,Aaaaaahhhh!,The Return of the King
2011,2011,PIPPIN,"It's the tree! Gandalf, Gandalf.",The Return of the King
982,982,FRODO,"We've lingered here too long.Come on, Sam.",The Two Towers


In [61]:
rt['movie'].value_counts()

movie
The Two Towers                 1010
The Return of the King          873
The Fellowship of the Ring      507
Name: count, dtype: int64

In [62]:
movies = pd.concat([df, rt], axis=1)
movies.sample(2)

,birth,death,gender,hair,height,name,race,realm,spouse,Unnamed: 0,char,dialog,movie
20,"Early ,First Age",NaN,Male,NaN,Tallest of the Elves of Gondolin,Pengolodh,Elves,NaN,NaN,20,FRODO,Gimli!,The Return of the King
549,NaN,NaN,NaN,NaN,NaN,Widow Rumble,NaN,NaN,NaN,549,PIPPIN,I get it. Don't be hasty.,The Two Towers


In [63]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2390 entries, 0 to 2389
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   birth       704 non-null    object
 1   death       595 non-null    object
 2   gender      768 non-null    object
 3   hair        172 non-null    object
 4   height      98 non-null     object
 5   name        911 non-null    object
 6   race        771 non-null    object
 7   realm       197 non-null    object
 8   spouse      465 non-null    object
 9   Unnamed: 0  2390 non-null   int64 
 10  char        2390 non-null   object
 11  dialog      2389 non-null   object
 12  movie       2390 non-null   object
dtypes: int64(1), object(12)
memory usage: 242.9+ KB


In [64]:
# Remove columns that are not needed for the recommendation system
movies.drop(columns=['birth','death','hair','height','spouse','Unnamed: 0'], inplace=True)

In [65]:
movies.head(4)

,gender,name,race,realm,char,dialog,movie
0,Female,Adanel,Men,NaN,DEAGOL,"Oh Smeagol Ive got one! , Ive got a fish Smeag...",The Return of the King
1,Male,Boromir,Men,NaN,SMEAGOL,"Pull it in! Go on, go on, go on, pull it in!",The Return of the King
2,Male,Lagduf,Orcs,NaN,DEAGOL,Arrghh!,The Return of the King
3,Male,Tarcil,Men,Arnor,SMEAGOL,Deagol!,The Return of the King


In [66]:
movies.shape

(2390, 7)

In [67]:
movies.duplicated().sum()

10

In [68]:
# Drop duplicate rows
movies.drop_duplicates(inplace=True)

In [69]:
movies.isnull().sum()

gender    1612
name      1469
race      1609
realm     2183
char         0
dialog       1
movie        0
dtype: int64

In [70]:
movies[movies['dialog'].isnull()]

,gender,name,race,realm,char,dialog,movie
1569,NaN,NaN,NaN,NaN,GIMLI,NaN,The Fellowship of the Ring


In [71]:
# Drop the row where dialog is missing
movies.dropna(subset=['dialog'], inplace=True)

In [72]:
# Fill missing categorical metadata with 'Unknown'
metadata_cols = ['gender', 'name', 'race', 'realm']
movies[metadata_cols] = movies[metadata_cols].fillna('Unknown')

In [73]:
movies.isnull().sum()

gender    0
name      0
race      0
realm     0
char      0
dialog    0
movie     0
dtype: int64

In [74]:
movies['dialog'][8]

"Because' , it's my birthday and I wants it. \xa0"

In [75]:
movies['dialog'][67]

"Salted pork' \xa0"

In [76]:
import re

In [77]:
def clean_text(text):
    for i in text:
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        text = re.sub(r'\s*,\s*', ' ', text)
        text = re.sub(r'\xa0', '', text)
    return text

In [78]:
movies['dialog']= movies['dialog'].apply(clean_text)

In [79]:
movies['dialog'][8]

'because  its my birthday and i wants it '

In [80]:
def preprocess_text(text):
    
    tokens = nltk.word_tokenize(text)
    tokens = [token for token in tokens if token.isalpha()]
    
    # Remove stopwords
    stopwords = set(nltk.corpus.stopwords.words('english'))
    tokens = [token for token in tokens if token not in stopwords]
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

In [81]:
movies['dialog'] = movies['dialog'].apply(preprocess_text)

In [82]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [83]:
def dialog_lemmatization(text):
    lemmatized_tokens = [lemmatizer.lemmatize(text)]
    lemmatized_text = ' '.join(lemmatized_tokens)
    return lemmatized_text

In [84]:
movies['dialog'] = movies['dialog'].apply(dialog_lemmatization)

In [85]:
movies['dialog'][0]

'oh smeagol ive got one ive got fish smeagol smeagol'

In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [87]:
tfidf = TfidfVectorizer(max_features=1000,ngram_range=(1, 2))

In [88]:
tfidf.fit_transform(movies['dialog']).toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [89]:
tfidf.get_feature_names_out()

array(['abyss', 'accept', 'ach', 'across', 'age', 'ago', 'ah', 'ahead',
       'aid', 'aim', 'air', 'ale', 'alive', 'almost', 'alone', 'along',
       'already', 'alright', 'also', 'always', 'ammen', 'amon', 'amongst',
       'andelu', 'andelu ven', 'another', 'another one', 'answer',
       'anymore', 'anything', 'aragorn', 'aragorn son', 'arm', 'armed',
       'armies', 'armies mordor', 'army', 'around', 'arwen', 'ash',
       'ash nazg', 'ask', 'attack', 'authority', 'awaits', 'away', 'aye',
       'back', 'bad', 'baggins', 'baggins baggins', 'barad', 'battle',
       'beacons', 'bear', 'beauty', 'began', 'begin', 'behind', 'believe',
       'belong', 'belongs', 'best', 'beth', 'betrayed', 'better',
       'beyond', 'big', 'bilbo', 'bilbo baggins', 'bilbos', 'birthday',
       'bit', 'black', 'black gate', 'blade', 'blade broken', 'blood',
       'boe', 'borders', 'born', 'boromir', 'boromir would', 'bound',
       'boy', 'brace', 'brace gate', 'brave', 'breached', 'bread',
       '

In [90]:
tfidf.idf_

array([7.38856141, 6.9830963 , 7.38856141, 6.9830963 , 6.69541422,
       6.82894562, 6.82894562, 6.69541422, 7.16541785, 7.38856141,
       6.9830963 , 7.16541785, 6.28994912, 6.9830963 , 5.83041679,
       6.69541422, 6.47227067, 6.69541422, 6.82894562, 5.59680194,
       7.16541785, 7.38856141, 7.38856141, 7.38856141, 7.38856141,
       6.13579844, 7.38856141, 6.28994912, 7.38856141, 6.47227067,
       5.30911986, 7.38856141, 7.38856141, 7.16541785, 6.47227067,
       7.16541785, 6.13579844, 6.69541422, 6.69541422, 7.16541785,
       8.08170859, 6.47227067, 6.69541422, 7.38856141, 7.38856141,
       5.68381331, 7.16541785, 4.64772138, 6.82894562, 5.94164242,
       8.08170859, 7.38856141, 5.88448401, 7.38856141, 6.9830963 ,
       7.38856141, 6.9830963 , 6.69541422, 6.28994912, 6.37696049,
       7.16541785, 7.38856141, 6.82894562, 7.38856141, 7.16541785,
       6.69541422, 6.82894562, 6.9830963 , 5.83041679, 6.82894562,
       7.16541785, 7.16541785, 6.69541422, 6.28994912, 6.98309

In [91]:
# shape of the tfidf matrix
tfidf_matrix = tfidf.transform(movies['dialog'])
print(tfidf_matrix.shape)

(2379, 1000)


In [92]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

print(similarity)
print(similarity.shape)

[[1.         0.         0.         ... 0.         0.19360953 0.        ]
 [0.         1.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.1842437 ]
 [0.19360953 0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.1842437  0.         1.        ]]
(2379, 2379)


In [93]:
similarity[0]

array([1.        , 0.        , 0.        , ..., 0.        , 0.19360953,
       0.        ])

## Explain Why Cosine Similarity Is Used

Cosine similarity is used to measure how similar two character dialogues based on their TF-IDF vectors. 

It compares the direction of vectors rather than their magnitude, making it useful for text similarity. 

It helps identify characters with similar dialogue vocabulary and recommend the top 5 most similar characters.


In [95]:
query = dialog_lemmatization(preprocess_text(clean_text("I am Gandalf the Grey")))
query_vector = tfidf.transform([query])

if query_vector.nnz == 0:
    raise ValueError("The query has no terms in the fitted TF-IDF vocabulary.")

scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
movie_index = int(scores.argmax())  # Position aligned with tfidf_matrix and similarity

print(movies.iloc[movie_index][['char', 'dialog']])

char                     ARAGORN
dialog    im friend gandalf grey
Name: 1441, dtype: object


In [113]:
def recommend(dialog, top_n=5):
    

    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
    results = []
    seen_chars = set()

    for idx in scores.argsort()[::-1]:
        char = movies.iloc[idx]["char"]
        if scores[idx] > 0 and char not in seen_chars:
            results.append(char)
            seen_chars.add(char)
            if len(results) >= top_n:
                break

    return results  

In [114]:
recommend("Well, starve, then. And good riddance!", top_n=5)

['ARAGORN', 'GRIMA', 'FRODO', 'BILBO', 'FRODO VOICE OVER']

In [115]:
recommend("Where is my son?	", top_n=5)

['ARAGORN', 'GRIMA', 'FRODO', 'BILBO', 'FRODO VOICE OVER']

In [109]:
import pickle
with open('movies.pkl','wb') as f:
    pickle.dump(movies,f)

In [110]:
import joblib
joblib.dump(similarity, 'similarity.pkl')

['similarity.pkl']